# Read and write JSON Lines with rugo

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mabel-dev/rugo.dev/blob/main/docs/notebooks/jsonl.ipynb)

Rugo reads and writes JSON Lines (one JSON object per line) with no PyArrow, pandas or NumPy. Fields you don't ask for are skipped without being parsed into values, and rows are filtered as they're read.

Every example uses the same dataset: 4,630 orbital launch attempts, from Sputnik onwards. More at [rugo.dev](https://rugo.dev).

In [ ]:
%pip install -q rugo

In [ ]:
import os
import urllib.request

if not os.path.exists("space_missions.jsonl"):
    urllib.request.urlretrieve("https://raw.githubusercontent.com/mabel-dev/rugo.dev/main/docs/data/space_missions.jsonl", "space_missions.jsonl")
print(f"space_missions.jsonl: {os.path.getsize('space_missions.jsonl'):,} bytes")

## Read the schema

`read_metadata` counts the lines and infers each field's type from the first few rows - which, as the next step shows, isn't always enough.

In [ ]:
from rugo import jsonl

meta = jsonl.read_metadata("space_missions.jsonl")
print(meta.num_rows, "rows")
for column in meta.schema_columns:
    print(column)

## Declare types inference can't see

Types are inferred from a small sample - here the first launches have no price, so `Price` looks like text. `explicit_schema` declares a field's type; the launch time is declared as a timestamp too.

In [ ]:
SCHEMA = {"Price": "FLOAT64", "Lauched_at": "TIMESTAMP[us]"}

with jsonl.read_jsonl("space_missions.jsonl", explicit_schema=SCHEMA) as reader:
    morsel = next(iter(reader))

for name, kind in zip(morsel.column_names, morsel.column_types):
    print(f"{name.decode():16} {kind}")
print(morsel)

## Read only what you need

`columns` picks fields; `predicates` filters rows as they're read. Declare types only for the fields you read.

In [ ]:
with jsonl.read_jsonl(
    "space_missions.jsonl",
    columns=["Company", "Mission", "Price"],
    predicates=[("Company", "==", "SpaceX")],
    explicit_schema={"Price": "FLOAT64"},
) as reader:
    spacex = next(iter(reader))

print(spacex.num_rows, "SpaceX launches")
print(spacex)

## Get Python values

A morsel holds typed columns. `column(name).to_pylist()` turns one into a Python list - only do this for the columns you need, since it's the step that creates a Python object per value.

In [ ]:
prices = [p for p in spacex.column("Price").to_pylist() if p is not None]
print(len(prices), "with a price; cheapest", min(prices), "most expensive", max(prices), "($M)")

## Write JSON Lines

`write_jsonl` turns a morsel into JSON Lines bytes. Read it back to check it round-trips.

In [ ]:
with open("spacex.jsonl", "wb") as f:
    f.write(jsonl.write_jsonl(spacex))

print(open("spacex.jsonl").readline())
print(jsonl.read_metadata("spacex.jsonl").num_rows, "rows read back")